# Capstone Project: Slogan Classifier and Generator

In this capstone project you will train a Long Short-Term Memory (LSTM) model to generate slogans for businesses based on their industry, and also train a classifier to predict the industry based on a given slogan.

##Libraries
We recommend running this notebook using [Google Colab](https://colab.google/) however if you choose to use your local machine you will need to install spaCy before starting.

To install spaCy, refer to the installation instructions provided on the spaCy [website](https://spacy.io/usage). Note you may need to install an older version of Python that is compatible with spaCy. You can create a virtual environment for this project to install the specific version of Python that you need.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
import spacy # available on Google Colab
from sklearn.model_selection import train_test_split

## Loading and viewing the dataset

- Load the slogan dataset into a variable called data.
- Extract relevant columns in a variable called df.
- Handle missing values.

Do **not** change the column names.

If you are using Google Colab you will need mount your Google Drive as follows:  
`from google.colab import drive`  
`drive.mount('/content/drive')`  

The path you use when loading your data will look something like this if you are using your Google Drive:  
"/content/drive/MyDrive/Colab Notebooks/slogan-valid.csv"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Load data
data = pd.read_csv("slogan-valid.csv")

# Extract relevant columns
df = data[["output", "industry"]].copy()

# Drop rows with missing values in either column
df.dropna(subset=["output", "industry"], inplace=True)

print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nSample rows:")
df.head()

Shape: (5346, 2)

Missing values:
output      0
industry    0
dtype: int64

Sample rows:


,output,industry
0,Taking Care of Small Business Technology,computer hardware
1,Build World-Class Recreation Programs,"health, wellness and fitness"
2,Most Powerful Lead Generation Software for Mar...,internet
3,Hire quality freelancers for your job,internet
4,"Financial Advisers Norwich, Norfolk",financial services


## Data Preprocessing

Since we are working with textual data, we need software that understands natural language. For this, we'll use a library for processing text called **spaCy**. Using spaCy, we'll break the text into smaller units called tokens that are easier for the machine to process. This process is called **tokenisation**. We'll also convert all text to lowercase and remove punctuation because this information is not necessary for our models. Run the code below, and your dataframe (df) will gain a new column called **'processed_slogan'** which contains the preprocessed text.




In [5]:
# Load spaCy model for text processing
nlp = spacy.load("en_core_web_sm")

# Define text preprocessing function
def preprocess_text(text):
    text_lower = text.lower()
    doc = nlp(text_lower)

    processed_tokens = []

    for token in doc:
        if not token.is_punct:
            processed_tokens.append(token.text)

    return " ".join(processed_tokens)

df["processed_slogan"] = df["output"].apply(preprocess_text)

df.head()

,output,industry,processed_slogan
0,Taking Care of Small Business Technology,computer hardware,taking care of small business technology
1,Build World-Class Recreation Programs,"health, wellness and fitness",build world class recreation programs
2,Most Powerful Lead Generation Software for Mar...,internet,most powerful lead generation software for mar...
3,Hire quality freelancers for your job,internet,hire quality freelancers for your job
4,"Financial Advisers Norwich, Norfolk",financial services,financial advisers norwich norfolk


We want our model to generate **industry-specific** slogans. If we use the 'processed_slogan' column as it is, we'll be leaving out crucial context - the industries of the companies behind those slogans. To fix this, we'll create a new **'modified_slogan'** column that adds the industry name to the front of processed slogan.  

For example:  

> industry = 'computer hardware'  
processed_slogan = 'taking care of small business technology'  
modified_slogan = 'computer hardware taking care of small business technology'

Write code in the cell below to achieve this.

In [6]:
# Prepend the industry name to the processed slogan to give the model industry context
df["modified_slogan"] = df["industry"] + " " + df["processed_slogan"]

print("Sample modified slogans:")
df[["industry", "processed_slogan", "modified_slogan"]].head(3)

Sample modified slogans:


,industry,processed_slogan,modified_slogan
0,computer hardware,taking care of small business technology,computer hardware taking care of small busines...
1,"health, wellness and fitness",build world class recreation programs,"health, wellness and fitness build world class..."
2,internet,most powerful lead generation software for mar...,internet most powerful lead generation softwar...


Now we need to get data to train our model. We have textual data which we will need to represent numerically for our model to learn from it.  
The code below does the following:
1. Tokenizes a dataset of slogans.
2. Converts words to numerical indices.
3. Creates input sequences using the numerical indices.  

Here's how it works. From the 'modified_slogan' column, we take the slogan "computer hardware taking care of small business technology". The tokenisation process will convert words into their corresponding indices:  

<center>

| Word         | Token Index |
|-------------|-------|
| "computer"  | 1     |
| "hardware"  | 2     |
| "taking"    | 3     |
| "care"      | 4     |
| "of"        | 5     |
| "small"     | 6     |
| "business"  | 7     |
| "technology"| 8     |

</center>

So the tokenized list is:

<center>
[1, 2, 3, 4, 5, 6, 7, 8]
</center>

When creating input sequences for training, the loop generates progressively longer sequences.

<center>

| Token Index Sequence               | Corresponding Slogan                                 |
|------------------------------|-----------------------------------------------------|
| [1, 2]                       | "computer hardware"                                |
| [1, 2, 3]                    | "computer hardware taking"                        |
| [1, 2, 3, 4]                 | "computer hardware taking care"                   |
| [1, 2, 3, 4, 5]              | "computer hardware taking care of"                |
| [1, 2, 3, 4, 5, 6]           | "computer hardware taking care of small"          |
| [1, 2, 3, 4, 5, 6, 7]        | "computer hardware taking care of small business" |
| [1, 2, 3, 4, 5, 6, 7, 8]     | "computer hardware taking care of small business technology" |

</center>

Instead of training the model on only **complete slogans**, we provide partial phrases which will help the model learn how words connect over time. This will make it better at predicting the next word when generating slogans.  

Run the cell block below to generate the input sequences. Be sure to read the comments to understand what the code is doing.


In [7]:
# Tokenizer to convert words into numerical values tokens
tokenizer = Tokenizer()

# Tokenizer learns words in dataset
tokenizer.fit_on_texts(df["modified_slogan"])

# Total number of unique words in learned vocabulary
total_words = len(tokenizer.word_index) + 1

# Dictionary mapping words to its numerical index: index based on frequency i.e., more freq => lower index
tokenizer.word_index

# Creating input sequences
# Initialise list to store the input sequences
input_sequences = []

# Iterate over processed slogans
for line in df["modified_slogan"]:

    # Convert slogans to token sequences
    token_list = tokenizer.texts_to_sequences([line])[0] # returns list containing list of words indices; extracting inner list [0]

    # token_list is a list of tokenized word INDICES
    # Building list of progressively longer input sequences for better training
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

The input sequences created above are of **varying lengths**, which will be a problem when training our LSTM model. LSTMs require input sequences of **equal length**. So, we need to **pad** shorter sequences by **prepending zeros** until they match the length of the longest sequence.  

For example, if the longest sequence has **10 tokens**, our padded sequences will look like this:

<center>

| Input Sequence                     | Padded Sequence                         |
|-------------------------------------|-----------------------------------------|
| [1, 2]                              | [0, 0, 0, 0, 0, 0, 0, 0, 1, 2]         |
| [1, 2, 3]                           | [0, 0, 0, 0, 0, 0, 0, 1, 2, 3]         |
| [1, 2, 3, 4]                        | [0, 0, 0, 0, 0, 0, 1, 2, 3, 4]         |
| [1, 2, 3, 4, 5]                     | [0, 0, 0, 0, 0, 1, 2, 3, 4, 5]         |
| [1, 2, 3, 4, 5, 6]                  | [0, 0, 0, 0, 1, 2, 3, 4, 5, 6]         |
| [1, 2, 3, 4, 5, 6, 7]               | [0, 0, 0, 1, 2, 3, 4, 5, 6, 7]         |
| [1, 2, 3, 4, 5, 6, 7, 8]            | [0, 0, 1, 2, 3, 4, 5, 6, 7, 8]         |

</center>

In the cell below, write code that **finds the length of the longest sequence** in **input_sequences** and stores this value in a variable named **max_seq_len**.


In [8]:
# Find the length of the longest sequence in input_sequences
max_seq_len = max(len(seq) for seq in input_sequences)

print(f"Maximum sequence length: {max_seq_len}")

Maximum sequence length: 15


Run the cell below to pad the input sequences so they are all the same length as **max_seq_length**.

In [9]:
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding="pre")

## Training Data for Slogan Generator

The input sequences generated will be used as our training data. Our LSTM needs to learn how to predict the **next word** in a sequence.  

The inputs for our model will be the input sequences **excluding the last token index** and the outputs will be the **last token index**.  

As an example, let us use the input sequence [0, 0, 1, 2, 3, 4, 5, 6, 7, 8] and say it corresponds to the slogan "computer hardware taking care of small business technology". When training the model:

> Our input **x** will be the input sequence [0, 0, 1, 2, 3, 4, 5, 6, 7] corresponding to "computer hardware taking care of small".  
> Our output **y** will be [8] which corresponds to "business".  

In the code cell below, use `input_sequences` to create the following two variables:
1. **X_gen** which contains the input sequences excluding the last token index.
2. **y_gen** which contains the last token index of the input sequence.

In [10]:
# All tokens except the last form the input
X_gen = input_sequences[:, :-1]

# The last token is the target (next word to predict)
y_gen = input_sequences[:, -1]

print(f"X_gen shape: {X_gen.shape}")
print(f"y_gen shape: {y_gen.shape}")

X_gen shape: (34736, 14)
y_gen shape: (34736,)


The model will output the next word of a sequence over a probability distribution. We need to encode our output variable for this to be possible.

In the code cell below, write code that will apply one-hot encoding to **y_gen** using `tf.keras.utils.to_categorical()`. **Maintain the same variable name**.  

*Hint: set the `num_classes` (number of classes) parameter to the total number of unique words in the learned vocabulary. You can access this value through a variable that was created when generating input sequences earlier.*

In [11]:
# One-hot encode y_gen so the model can output a probability distribution over the vocabulary
y_gen = tf.keras.utils.to_categorical(y_gen, num_classes=total_words)

print(f"y_gen shape after one-hot encoding: {y_gen.shape}")

y_gen shape after one-hot encoding: (34736, 6046)


## Slogan Generator Architecture

In the code cell that follows, configure the LSTM following these steps:

1. Create a sequential model using `tf.keras.models.Sequential()`. This model will have an embedding layer, two LSTM layers, and a dense output layer.
2. Add an embedding layer that converts words into dense vector representations. This layer should:
> *   Have `total_words`as the vocabulary size.
> *   Use 100 as an embedding dimension.
> *   Takes an input length of `max_seq_len - 1` (excludes the target word).
3. Add two LSTM layers.
> *   The first LSTM layer should have 150 **and** set `return_sequences` to `True`.
> *   The second LSTM layer should have 100 units.
4. Add a dense output layer which:
> *   Uses `total_words` as the number of units (one for each word in the vocabulary).
> *   Uses a softmax activation function.
5. Use `Sequential` to put everything together in the correct order to complete the architecture of the LSTM model called **gen_model**.


In [12]:
gen_model = tf.keras.models.Sequential([
    # Embedding layer: maps each word index to a dense 100-dimensional vector
    Embedding(total_words, 100, input_length=max_seq_len - 1),

    # First LSTM layer: returns full sequence so the second LSTM can process it
    LSTM(150, return_sequences=True),

    # Second LSTM layer: processes the sequence and returns a single output vector
    LSTM(100),

    # Dense output layer: one unit per word in the vocabulary with softmax probabilities
    Dense(total_words, activation="softmax")
])

gen_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In the code cell below, compile `gen_model` using `categorical_crossentropy` loss, an Adam optimiser, and an appropriate metric of your choice.


In [13]:
gen_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

## Slogan Generation

In the code cell below, fit the compiled model on the inputs and outputs, setting the **number of epochs to 50**.

In [14]:
gen_history = gen_model.fit(
    X_gen, y_gen,
    epochs=50,
    verbose=1
)

Epoch 1/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 77s 67ms/step - accuracy: 0.0696 - loss: 7.0276
Epoch 2/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 74s 68ms/step - accuracy: 0.1078 - loss: 6.3561
Epoch 3/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 74s 68ms/step - accuracy: 0.1425 - loss: 6.0259
Epoch 4/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 81s 67ms/step - accuracy: 0.1760 - loss: 5.7530
Epoch 5/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 73s 67ms/step - accuracy: 0.1988 - loss: 5.5036
Epoch 6/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 73s 68ms/step - accuracy: 0.2174 - loss: 5.2720
Epoch 7/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 74s 68ms/step - accuracy: 0.2335 - loss: 5.0486
Epoch 8/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 73s 67ms/step - accuracy: 0.2441 - loss: 4.8347
Epoch 9/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 73s 67ms/step - accuracy: 0.2561 - loss: 4.6275
Epoch 10/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 72s 66ms/step - accuracy: 0.2662 - loss: 4.4260
Epoch 11/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 73s 68ms/step - accuracy: 0.2742 - loss: 4.2256
Epoch 12

We will now define a function called `generate_slogan` which will generate a slogan by predicting one word at a time based on a given starting phrase (the `seed_text`). This function will do this using our trained model, `gen_model`.

Here is a breakdown of how the algorithm works:  

Let us assume the dictionary mapping words to unique indices, `tokenizer.word_index`, looks like this:

> `{'computer': 1, 'hardware': 2, 'taking': 3, 'care': 4, 'of': 5}`

If the model's predicted index for the next word is 3 (`predicted_index = 3`), the loop will:

> Check 'computer' (index 1) → No match  
> Check 'hardware' (index 2) → No match  
> Check 'taking' (index 3) → Match found!  
> Assign output_word = "taking" and exit the loop.  

The `output_word` will be appended to the `seed_text`, and the process will continue to add words to the `seed_text` until we have reached the maximum number of words **or** an invalid prediction occurs.  

Carefully follow the code below and complete the missing parts as guided by the comments.

In [15]:
def generate_slogan(seed_text, max_words=20):
    for _ in range(max_words):

        # Tokenise and pad the current seed text
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding="pre")

        # Predict the probability distribution of the next word over the vocabulary
        predictions = gen_model.predict(token_list, verbose=0)

        # Select the index with the highest predicted probability
        predicted_index = np.argmax(predictions)

        output_word = None

        # Find the word that corresponds to the predicted index
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        # Stop if no valid word is found
        if output_word is None:
            break

        # Append the predicted word to the seed text and continue
        seed_text += " " + output_word

    return seed_text


# Test the generator with a sample industry
print(generate_slogan("internet"))

internet web design christchurch digital marketing strategies for start ups leaders south professionals apps oh professionals projects andrews communications odessa services


## Training Data for Slogan Classifier

We will now prepare the data we will use to train our classifier. For our classifier, the inputs will come from the `processed_slogans` column of our DataFrame, `df`. The outputs will be the different industry categories under the `industry` column.

In the code cell below, extract the unique values from the `industry` column in the DataFrame and store these in a variable called **industries**.

In [16]:
# Extract unique industry values from the DataFrame
industries = df["industry"].unique()

print(f"Total unique industries: {len(industries)}")
print(industries[:10])

Total unique industries: 142
['computer hardware' 'health, wellness and fitness' 'internet'
 'financial services' 'mechanical or industrial engineering'
 'marketing and advertising' 'hospital & health care' 'research'
 'information technology and services' 'computer software']


Create a dictionary called `industry_to_index` where each unique industry is mapped to a unique index starting from 0.

*Hint: Use the `enumerate()` function.*

In [17]:
# Map each industry to a unique integer index starting from 0
industry_to_index = {industry: idx for idx, industry in enumerate(industries)}

print("Sample mappings:")
for k, v in list(industry_to_index.items())[:5]:
    print(f"  {k!r}: {v}")

Sample mappings:
  'computer hardware': 0
  'health, wellness and fitness': 1
  'internet': 2
  'financial services': 3
  'mechanical or industrial engineering': 4


Create a new column `industry_index` in your DataFrame by mapping the `industry` column to the indices using the `industry_to_index` dictionary.

*Hint: Use the  `map()` function.*

In [18]:
# Create a new column mapping each row's industry to its integer index
df["industry_index"] = df["industry"].map(industry_to_index)

df[["industry", "industry_index"]].head()

,industry,industry_index
0,computer hardware,0
1,"health, wellness and fitness",1
2,internet,2
3,internet,2
4,financial services,3


Split the DataFrame `df` into training and testing sets, setting aside 20% of the data for the test set. Be sure to set the parameter `stratify=df["industry_index"]`. This ensures that both sets have the same proportion of each class (industry) as in the original dataset, resulting in balanced datasets. Call the training DataFrame `df_train` and the testing DataFrame `df_test`.

In [23]:
# Remove industries with only 1 sample (can't be stratified)
counts = df["industry_index"].value_counts()
df = df[df["industry_index"].isin(counts[counts >= 2].index)]

# Split into 80% training and 20% testing, stratified by industry
df_train, df_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["industry_index"]
)

print(f"Training set: {df_train.shape[0]} rows")
print(f"Testing set: {df_test.shape[0]} rows")

Training set: 4272 rows
Testing set: 1068 rows


Our classifier will use padded slogan sequences as inputs, similar to input sequences used for the slogan generator. The difference is we will not use sequences that get progressively longer, but instead we will use **complete slogans**. This is because our classifier does not need to learn how to predict what word comes next. It needs the full context of a slogan to learn how to accurately predict the industry.  

The next steps will walk you through how to create these sequences.  

We previously created and fitted a `Tokenizer` object called `tokenizer` while preparing data for the slogan generator. Now, we will reuse it to convert words into numerical indices.  

In the code cell below, use the `texts_to_sequences()` **method** of `tokenizer` to transform the `processed_slogan` column in **both** the `df_train` and `df_test` DataFrames into sequences of numerical indices. Store the results in variables named `X_train` and `X_test`.


In [24]:
# Convert processed slogans to sequences of numerical indices using the fitted tokenizer
X_train = tokenizer.texts_to_sequences(df_train["processed_slogan"])
X_test = tokenizer.texts_to_sequences(df_test["processed_slogan"])

print(f"X_train sequences: {len(X_train)}")
print(f"X_test sequences: {len(X_test)}")

X_train sequences: 4272
X_test sequences: 1068


The slogan sequences are of varying lengths. We will need to pad them the same way we did to the input sequences for the slogan generator. The `pad_sequences()` function can ensure the sequences in `slogan_sequences` have the same length.  

In the code cell below, use the `pad_sequences()` function to standardise the `slogan_sequences` lengths. Set the `maxlen` parameter to `max_seq_len`, the `padding` parameter to 0, and assign the resulting padded sequences to the same variables, `X_train` and `X_test`.

In [25]:
# Pad sequences to uniform length so all inputs are the same size
X_train = pad_sequences(X_train, maxlen=max_seq_len, padding="pre")
X_test = pad_sequences(X_test,  maxlen=max_seq_len, padding="pre")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (4272, 15)
X_test shape: (1068, 15)


We have successfully created training and testing inputs for our model. Now, we will create the outputs - industry categories.

 In the code cell that follows, use `tf.keras.utils.to_categorical()` to apply one-hot encoding to the `industry_index` column of **both** `df_train` and `df_test` DataFrames. Assign the results to a variables named `y_train` and `y_test`.

 *Hint: set the `num_classes` parameter to the total number of industries in the DataFrame. The `industries` variable can be used to find this value.*

In [26]:
# One-hot encode the industry index labels for both training and testing sets
y_train = tf.keras.utils.to_categorical(
    df_train["industry_index"], num_classes=len(industries)
)
y_test = tf.keras.utils.to_categorical(
    df_test["industry_index"], num_classes=len(industries)
)

print(f"y_train shape: {y_train.shape}")
print(f"y_test  shape: {y_test.shape}")

y_train shape: (4272, 142)
y_test  shape: (1068, 142)


## Slogan Classifier Architecture

Configure the LSTM classifier following these steps:  


1. Create a Sequential model:  
   Use `tf.keras.models.Sequential()` to create a sequential model. This model will consist of an embedding layer, two LSTM layers, and a dense output layer.

2. Add an embedding layer which will convert words into dense vector representations. Configure this layer with:
   > * `total_words` as the vocabulary size.
   > * 100 as the embedding dimension.
   > * `max_seq_len` as the `input_length` (this is the length of the slogans).

3. Add the first LSTM layer. Configure it with:
   > * 150 units.
   > * Set `return_sequences` to `True` to ensure the layer outputs sequences for the next LSTM layer.

4. Add the second LSTM layer which will process the output from the previous LSTM layer. Configure it with:
   > * 100 units.
   > * No need to set `return_sequences` here (it is the final LSTM layer).

5. Add the dense output layer which will classify the data into industries. Configure it with:
   > * The number of unique industries as the number of units.
   > * The `softmax` activation function to get probabilities for each class (industry).

6. Use `Sequential` to arrange all layers in the correct order and complete the architecture of the LSTM model called **class_model**.


In [27]:
class_model = tf.keras.models.Sequential([
    # Embedding layer: maps each word index to a 100-dimensional dense vector
    Embedding(total_words, 100, input_length=max_seq_len),

    # First LSTM layer: returns full sequence for the next LSTM layer
    LSTM(150, return_sequences=True),

    # Second LSTM layer: summarises the sequence into a single output vector
    LSTM(100),

    # Dense output layer: one unit per industry with softmax probabilities
    Dense(len(industries), activation="softmax")
])

class_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In the code cell below, compile `class_model` using `categorical_crossentropy` loss, an Adam optimiser, and an appropriate metric of your choice.

In [28]:
class_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

## Slogan Classification & Evaluation

In the code cell that follows, fit the compiled model on the inputs and outputs, setting **the number of epochs to 50**.

In [29]:
class_history = class_model.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 14s 78ms/step - accuracy: 0.0833 - loss: 4.3896 - val_accuracy: 0.0843 - val_loss: 4.2757
Epoch 2/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 58ms/step - accuracy: 0.0847 - loss: 4.2868 - val_accuracy: 0.0843 - val_loss: 4.2745
Epoch 3/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 9s 65ms/step - accuracy: 0.0878 - loss: 4.2688 - val_accuracy: 0.0843 - val_loss: 4.2462
Epoch 4/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.1081 - loss: 4.0778 - val_accuracy: 0.1180 - val_loss: 4.0678
Epoch 5/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 57ms/step - accuracy: 0.1934 - loss: 3.6465 - val_accuracy: 0.1564 - val_loss: 3.9220
Epoch 6/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 10s 72ms/step - accuracy: 0.2891 - loss: 3.1520 - val_accuracy: 0.1798 - val_loss: 3.8338
Epoch 7/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.3809 - loss: 2.7095 - val_accuracy: 0.1957 - val_loss: 3.9687
Epoch 8/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 10s 72ms/step - accuracy: 0.4618 - loss: 2.3480 - val_a

Evaluate the model using the testing set. Add a comment on the model's performance.

In [31]:
# Evaluate the classifier on the held-out test set
loss, accuracy = class_model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test Loss: 7.7228
Test Accuracy: 0.1816


We will now define a function called `classify_slogan` which takes a slogan as input and predicts the industry it belongs to using the trained model, `class_model`.  

Carefully follow the code below and complete the missing parts (indicated by ellipses) as guided by the comments.

In [32]:
def classify_slogan(slogan):
    # Preprocess the slogan using the same function applied during training
    slogan = preprocess_text(slogan)

    # Convert the slogan to a sequence of numerical indices
    sequence = tokenizer.texts_to_sequences([slogan])

    # Pad the sequence to match the input length expected by the classifier
    padded_sequence = pad_sequences(sequence, maxlen=max_seq_len, padding="pre")

    # Pass the padded sequence through the classifier to get industry probabilities
    prediction = class_model.predict(padded_sequence, verbose=0)

    # Select the index of the industry with the highest predicted probability
    predicted_index = np.argmax(prediction)

    # Return the industry name corresponding to the predicted index
    return industries[predicted_index]


# Test the classifier with a sample slogan
print(classify_slogan("Taking Care of Small Business Technology"))

computer hardware


## Combining the two models

Run the code cell below to combine the two models: we will first generate a slogan for a company in the "internet" industry, then pass the generated slogan to the slogan classifier to see if it correctly classifies it as internet.

In [33]:
industry = "internet"
generated_slogan = generate_slogan(industry)
predicted_industry = classify_slogan(generated_slogan)

print(f"Generated Slogan: {generated_slogan}")
print(f"Predicted Industry: {predicted_industry}")

Generated Slogan: internet web design christchurch digital marketing strategies for start ups leaders south professionals apps oh professionals projects andrews communications odessa services
Predicted Industry: semiconductors


Compare the results and comment on any differences you notice between the generated slogans and the classifier’s predictions in the markdown cell below.


**Comparison and Observations**

- The generator produced a slogan that started relevantly with "internet web design" and "digital marketing strategies" which language closely associated with the internet industry. However, it quickly deteriorated into a string of disconnected words such as "odessa services" and "andrews communications", suggesting the model began to lose coherence after the first few words.
- Despite the early industry-relevant language, the classifier predicted **semiconductors** rather than **internet**. This is likely because the slogan broke down into generic and unrelated terms, diluting the internet-specific signal the classifier needed to make a correct prediction.
- The classifier's test accuracy of **18.16%** reflects the difficulty of the task. With 142 industries to distinguish between and limited training data per class, the model struggles to generalise. A random baseline would achieve less than 1%, so 18% indicates the model has learned some signal, but performance is far from reliable.
- The high test loss of **7.72** further confirms that the classifier's predicted probability distributions are poorly calibrated, meaning it is often uncertain or wrong even when it arrives at a prediction.
- Overall, the pipeline reveals a compounding problem: the generator produces increasingly incoherent text beyond the first few words, and the classifier is not accurate enough to recover the correct industry even from partially relevant input. Both models would benefit from more training data per industry, more epochs, and stronger regularisation.